In [0]:
# Clean slate: drop the table and clear the checkpoint, so this run 
# starts fresh and correctly re-ingests all 9 real files
spark.sql("DROP TABLE IF EXISTS ledgr.bronze.sessions_raw_autoloader")

dbutils.fs.rm("s3://ledgr-raw-data-2026/checkpoints/bronze_autoloader/", recurse=True)
dbutils.fs.rm("s3://ledgr-raw-data-2026/schema/bronze_autoloader/", recurse=True)

print("Cleared stale table and checkpoint state")

In [0]:
from pyspark.sql import functions as F

checkpoint_path = "s3://ledgr-raw-data-2026/checkpoints/bronze_autoloader/"
schema_location = "s3://ledgr-raw-data-2026/schema/bronze_autoloader/"

autoloader_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_location)
    .load("s3://ledgr-raw-data-2026/raw/")
)

query = (autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("ledgr.bronze.sessions_raw_autoloader")
)

query.awaitTermination()
print("Auto Loader ingestion complete")

count = spark.table("ledgr.bronze.sessions_raw_autoloader").count()
print("Row count: " + str(count))
print("Expected: 10056 (matches the 9 real source files)")

# This notebook demonstrates Auto Loader's incremental ingestion capability
# as a proof of concept, separate from the main Bronze table
# (ledgr.bronze.sessions_raw), which uses a plain batch read since the
# real dataset is static.

# Incrementality was verified once: a 10th test file (a duplicate of
# 0000.parquet) was added to the raw/ folder, and rerunning this notebook
# confirmed the row count increased by exactly 150 (that file's row count),
# not by reprocessing all 10,056 original rows. That test file has since
# been removed from S3, so this table currently reflects only the 9 real
# source files (10,056 rows).